# Step 03 — ALKIS refinement

Refine the raw ALKIS/LoD2 layer into something worth carrying downstream. This
notebook is **structural only** — it changes what the rows and columns are, never
what they mean. Semantics (the AdV function codelist, the activity map, the OSM
POI join) are step 04's job.

| | |
|---|---|
| **Reads** | `data/output/02_alkis_lod2_raw.gpkg` (5.07 GB, `MultiPolygon Z`) |
| **Writes** | `data/output/03_alkis_buildings.gpkg` — one row per LoD2 **part** |
| | `data/output/03_alkis_by_building.gpkg` — one row per **ALKIS building** |
| | `data/experimental_extract/alkis_outlines.shp` — throwaway QGIS layer |
| **Needs** | `sqlite3` (stdlib), `pyogrio`, `shapely >= 2` |
| **Runtime** | ~10 min: the 5 GB 3D read, the surface classification and the dissolve dominate |

## What this notebook is

**The hand-off from 3D to 2D.** It takes 4,891,343 LoD2 *surfaces* and produces
1,332,029 flat building footprints (1,385,265 before the sliver filter), each carrying the 3D information as numbers
rather than as geometry. Notebook 04 enriches these with OSM.

**Two layers come out of it, and the part-level one is authoritative.**

The primary output has one row per **LoD2 part** (`gml_id`). Every height and
roof attribute is a property of the part — `height_top_m`, `roof_shape`,
`elev_top_m` all differ between the parts of one building — so collapsing to
`alkis_id` forces an arbitrary winner and loses the rest irreversibly.

Section 8 then also writes the collapsed version, one row per **ALKIS building**
(869,316), because that is what a person means by "a building" and what the POI
join in step 04 mostly wants. Both are kept: part → building is a `groupby`,
building → part is impossible.

| section | |
|---|---|
| 1–4 | Column triage: profile all 27, drop the 7 that carry no information |
| 5 | Classify every surface as `GROUND` / `WALL` / `ROOF` using Z |
| 6 | Build the part layer: footprint, both volumes, every surviving attribute |
| 7 | Write `03_alkis_buildings.gpkg` (part level) |
| 8 | Aggregate to ALKIS buildings and write `03_alkis_by_building.gpkg` |

Deliberately **not** done here: no volume threshold, no merging of adjacent
polygons, no function-code labels. Those need either the OSM layer or a decision
this notebook has no evidence for.

## Why SQL and not geopandas

The raw layer is 4,891,343 rows and 5.07 GB, most of it geometry. Loading the
attributes into pandas costs GBs of RAM — `Eigentum` and `Lizenz` alone are
~200 bytes on each of 4.9 M rows. A GeoPackage *is* a SQLite database, so the
profile below is one aggregate query against it: no geometry is read and nothing
is materialised in Python.

One pass with all 109 aggregates takes ~70 s. Computing them column by column
instead means 54 full scans of a 5.07 GB file — about 10 minutes — because each
scan pulls the geometry pages too whether you asked for them or not.

In [1]:
import os, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
# Same reason as steps 01 and 02: a notebook's working directory is not
# necessarily its own folder, so Path('..') is unreliable.
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError(
        'Cannot find the pipeline root (the folder containing config.py). '
        f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.'
    )
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# --- point GDAL/PROJ at this env's data files ---------------------------------
_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import sqlite3, time, warnings
import pandas as pd
import pyogrio

from config import (
    ALKIS_RAW_FILE, ALKIS_DROP_CONSTANT_COLS, TARGET_CRS,
    EXPERIMENTAL_DIR, ALKIS_OUTLINES_SHP,
    ALKIS_MIN_PART_AREA_M2, ALKIS_KEEP_LARGEST_PART_PER_BUILDING,
    ALKIS_BUILDINGS_FILE, ALKIS_RENAME, ALKIS_OUTPUT_COLS,
    ALKIS_ROOF_SPILL_MIN_WIDTH_M, ALKIS_ROOF_FALLBACK,
    ALKIS_BY_BUILDING_FILE, ALKIS_BUILDING_COLS, ALKIS_ALLOW_EMPTY_COLS,
)
from lib.checks import require_file, require_non_empty, require_unique
from lib.schema import assert_no_empty_columns

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

print('Root :', ROOT_DIR)
print('Raw  :', ALKIS_RAW_FILE.name)

Root : C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-FINAL
Raw  : 02_alkis_lod2_raw.gpkg


## 1. Input contract

The raw layer must exist and must still be the thing step 02 described: one row
per LoD2 *surface*, so `gml_id` repeats. Nothing here assumes otherwise.

`mode=ro` on the SQLite connection is deliberate — this notebook must not be
able to modify the source, and read-only also means it can be opened while QGIS
has the layer loaded.

**Read-only does not mean lock-free.** On Windows an open handle blocks
*deletion* of a file even when opened read-only, while the file still reports as
writable throughout. So the connection is closed explicitly at the end of
section 4. Left open, this kernel pins the layer and step 02 can never rewrite
it, failing with `[WinError 32] being used by another process` — which looks
exactly like a QGIS lock and is not one. If a step 02 rerun ever reports that
lock and QGIS is closed, restart this notebook's kernel.

In [2]:
require_file(ALKIS_RAW_FILE, 'raw ALKIS/LoD2 layer')

info = pyogrio.read_info(ALKIS_RAW_FILE, layer='buildings')
print(f'  ..  rows     : {info["features"]:,}')
print(f'  ..  geometry : {info["geometry_type"]}')
print(f'  ..  CRS      : {info["crs"]}')
if info['crs'] != TARGET_CRS:
    raise AssertionError(f'raw layer is {info["crs"]}, expected {TARGET_CRS}')

# A GeoPackage is a SQLite database. Read-only, so this cannot touch the source
# and does not fight QGIS for the file.
con = sqlite3.connect(f'file:{ALKIS_RAW_FILE}?mode=ro', uri=True)
cur = con.cursor()

schema = list(cur.execute('PRAGMA table_info(buildings)'))
GEOM_COL = 'geom'
ATTRS = [r[1] for r in schema if r[1] not in ('fid', GEOM_COL)]
print(f'  ..  columns  : {len(ATTRS)} attributes + fid + {GEOM_COL}')

N = cur.execute('SELECT COUNT(*) FROM buildings').fetchone()[0]
if N != info['features']:
    raise AssertionError(f'SQLite sees {N:,} rows, GDAL sees {info["features"]:,}')
print(f'  ok  {N:,} surface rows, agreed by SQLite and GDAL')

  ok  02_alkis_lod2_raw.gpkg (5,074.6 MB)
  ..  rows     : 4,891,343
  ..  geometry : MultiPolygon Z
  ..  CRS      : EPSG:25832
  ..  columns  : 27 attributes + fid + geom


  ok  4,891,343 surface rows, agreed by SQLite and GDAL


## 2. Profile every column

For each of the 27 attributes: how many rows are filled, how many distinct
values exist, and the value range. That is 4 aggregates × 27 columns + the row
count = 109 aggregates, all in **one** query so the file is scanned once.

`COUNT(col)` ignores NULLs while `COUNT(*)` does not, which is what makes the
fill rate fall out of the same pass. `COUNT(DISTINCT col)` is the column that
actually decides things here: **1 distinct value means the column cannot
distinguish any building from any other.**

In [3]:
parts = ['COUNT(*)']
for c in ATTRS:
    parts += [f'COUNT("{c}")', f'COUNT(DISTINCT "{c}")',
              f'MIN("{c}")', f'MAX("{c}")']

print(f'Profiling {len(ATTRS)} columns in one pass over {N:,} rows '
      f'(~70 s) ...', flush=True)
t0 = time.perf_counter()
row = cur.execute('SELECT ' + ', '.join(parts) + ' FROM buildings').fetchone()
print(f'  ok  one pass, {len(parts)} aggregates  [{time.perf_counter() - t0:,.1f}s]')

recs = []
for i, c in enumerate(ATTRS):
    filled, distinct, lo, hi = row[1 + i * 4: 5 + i * 4]
    numeric = isinstance(lo, (int, float)) and isinstance(hi, (int, float))
    recs.append({
        'column':   c,
        'filled':   filled,
        'fill_pct': round(100.0 * filled / N, 1),
        'distinct': distinct,
        'detail':   (f'{lo:,.3f} .. {hi:,.3f}' if numeric else str(lo)[:44]),
    })

profile = pd.DataFrame(recs)
print()
print(profile.to_string(index=False))

Profiling 27 columns in one pass over 4,891,343 rows (~70 s) ...


  ok  one pass, 109 aggregates  [76.4s]

    column  filled  fill_pct  distinct                                       detail
measHeight 4891343     100.0     24203                             0.000 .. 317.066
    gml_id 4891343     100.0   1385279                             DENILD01000000Fg
 externRef 4891343     100.0    869327 http://repository.gdi-de.org/schemas/adv/cit
  function 4891343     100.0        88                                   31001_1000
    DqDach 4891343     100.0         2                       1,000.000 .. 6,000.000
    DqLage 4891343     100.0         1                       1,000.000 .. 1,000.000
   DqBoden 4891343     100.0         1                       1,300.000 .. 1,300.000
creationDa 4891343     100.0        27                                   2020-11-18
GrundrissA 4818594      98.5        13                                   2017-08-01
LetzteAend 4891343     100.0         1                                   2024-09-30
 Geom2DRef 4891343     100.0       

## 3. Triage — what goes

**The rule for this step is narrow on purpose: a column goes only if it has
exactly one distinct value across all 4.9 M rows.** That is not a judgement
call, it is arithmetic — such a column cannot tell two buildings apart, so
nothing downstream can ever use it.

Everything else stays for now, including the two that *look* droppable:

* `Name` is only 2.0 % filled, but those ~99 k rows are the named buildings —
  halls, schools, venues. Sparse is not useless; that is exactly the population
  step 04 wants to match against OSM names.
* `Strasse` / `HausNr` are 54 % filled. Half is a real limitation to know about
  before anyone plans address matching, but it is not grounds for dropping.

`ALKIS_DROP_CONSTANT_COLS` in `config.py` records the expected set with a reason
per column. The check below **recomputes it from the data and fails if the two
disagree** — a differing set means the source release changed and the stored
reasoning needs revisiting, rather than a stale list quietly dropping the wrong
thing.

In [4]:
constant = set(profile.loc[profile['distinct'] <= 1, 'column'])
declared = set(ALKIS_DROP_CONSTANT_COLS)

if constant != declared:
    raise AssertionError(
        'the constant columns found in the data do not match '
        'ALKIS_DROP_CONSTANT_COLS in config.py.\n'
        f'  only in data   : {sorted(constant - declared)}\n'
        f'  only in config : {sorted(declared - constant)}\n'
        'The source release has changed - review the reasons in config.py '
        'before editing the list.'
    )

print(f'  ok  {len(constant)} constant columns, matching config.py exactly')
print()
print('DROP - one distinct value across all rows:')
for c in sorted(constant):
    val = profile.loc[profile['column'] == c, 'detail'].iloc[0]
    print(f'  {c:<12} = {val[:46]:<46}')
    print(f'  {"":<12}   {ALKIS_DROP_CONSTANT_COLS[c]}')

KEEP_COLS = [c for c in ATTRS if c not in constant]
print()
print(f'  ..  columns: {len(ATTRS)} -> {len(KEEP_COLS)}  '
      f'({len(constant)} dropped)')

# What the two bulky ones actually cost. Measured, not guessed: SQLite's
# length() is bytes for text, so this is the on-disk weight of the strings
# themselves, ignoring page overhead.
bulk = [c for c in ('Eigentum', 'Lizenz', 'externRef') if c in ATTRS]
if bulk:
    lens = cur.execute(
        'SELECT ' + ', '.join(f'length("{c}")' for c in bulk) +
        ' FROM buildings LIMIT 1').fetchone()
    print()
    print('  ..  weight of the long repeated strings:')
    total = 0
    for c, ln in zip(bulk, lens):
        mb = (ln or 0) * N / 1e6
        flag = 'DROPPED' if c in constant else 'kept for now'
        total += mb if c in constant else 0
        print(f'        {c:<11} {ln:>4} bytes x {N:,} rows = {mb:>7,.0f} MB   {flag}')
    print(f'        -> dropping the constant ones sheds ~{total:,.0f} MB of the '
          f'{ALKIS_RAW_FILE.stat().st_size / 1e6:,.0f} MB file')

  ok  7 constant columns, matching config.py exactly

DROP - one distinct value across all rows:
  DqBoden      = 1,300.000 .. 1,300.000                        
                 1300 - ground-surface quality flag, never varies
  DqLage       = 1,000.000 .. 1,000.000                        
                 1000 - positional quality flag, never varies
  Eigentum     = Landesamt fuer Geoinformation und Landesverm  
                 'Landesamt fuer Geoinformation ...' - the publisher, not the building
  Geom2DRef    = 3,000.000 .. 3,000.000                        
                 3000 - 2D geometry reference code, never varies
  Land         = Germany                                       
                 'Germany' - the whole extract is one country
  LetzteAend   = 2024-09-30                                    
                 2024-09-30 - one statewide edit date for the release
  Lizenz       = CC-BY-4.0, siehe https://creativecommons.org  
                 CC-BY-4.0 licence text, id

## 4. What survives, and what each column is for

The 20 survivors, grouped by the job they do. This is the map the remaining
steps work from — and the "decide next" group is exactly what steps 03.2 and
03.3 have to resolve.

Value breakdowns are printed for the low-cardinality survivors, because for
those the question "is this useful?" is answerable by just looking at the
values. They are read in a single pass for the same reason as the profile.

One discrepancy to expect rather than puzzle over: `GrundrissA` shows **13**
distinct in the profile above and **14** here. Both are right. SQL's
`COUNT(DISTINCT)` ignores NULL; the breakdown below uses
`value_counts(dropna=False)`, which counts "missing" as its own category. The
extra row is the 72,749 NULLs, and seeing them is the point.

In [5]:
ROLES = {
    'identity': ['gml_id'],
    'volume':   ['measHeight', 'Firsthoehe', 'Traufhoehe', 'AbsHoehe', 'DachFlaech'],
    'semantic': ['function', 'Name'],
    'location': ['AGS', 'Stadt', 'Strasse', 'HausNr'],
    'decide next': ['externRef', 'creationDa', 'GrundrissA', 'DqDach',
                    'roofType', 'DachName', 'DachNeig', 'DachOri'],
}

seen = set()
for role, cols in ROLES.items():
    here = [c for c in cols if c in KEEP_COLS]
    seen |= set(here)
    print(f'{role.upper()}  ({len(here)})')
    for c in here:
        r = profile.loc[profile['column'] == c].iloc[0]
        print(f'   {c:<12} {r["fill_pct"]:>6.1f}% filled  {r["distinct"]:>9,} distinct  '
              f'{r["detail"][:40]}')
    print()

unclassified = [c for c in KEEP_COLS if c not in seen]
if unclassified:
    print(f'!!  not classified above: {unclassified}')

# Low-cardinality survivors: one scan, then count in pandas.
small = [c for c in KEEP_COLS
         if 1 < profile.loc[profile['column'] == c, 'distinct'].iloc[0] <= 40]
print('=' * 72)
print(f'value breakdown of the {len(small)} low-cardinality survivors: {small}')
print('=' * 72)
t0 = time.perf_counter()
vals = pd.read_sql_query(
    'SELECT ' + ', '.join(f'"{c}"' for c in small) + ' FROM buildings', con)
print(f'  ..  read in one pass  [{time.perf_counter() - t0:,.1f}s]')

for c in small:
    vc = vals[c].value_counts(dropna=False)
    print(f'\n  {c}  ({len(vc)} distinct)')
    for v, k in vc.head(10).items():
        print(f'      {str(v)[:44]:<44} {k:>10,}  {100 * k / N:>5.1f}%')
    if len(vc) > 10:
        print(f'      ... and {len(vc) - 10} more')

# --- release the file ---------------------------------------------------------
# MUST close. An open SQLite handle blocks DELETION of the GeoPackage on Windows
# even when it was opened read-only, and the file still reports as writable the
# whole time. Left open, this kernel pins the multi-GB layer and step 02 can
# never rewrite it - it fails with `[WinError 32] being used by another process`,
# which reads exactly like a QGIS lock and is not one.
con.close()
print()
print('  ok  SQLite connection closed - the raw layer is no longer pinned')

IDENTITY  (1)
   gml_id        100.0% filled  1,385,279 distinct  DENILD01000000Fg

VOLUME  (5)
   measHeight    100.0% filled     24,203 distinct  0.000 .. 317.066
   Firsthoehe    100.0% filled    226,511 distinct  47.973 .. 1,053.975
   Traufhoehe    100.0% filled    223,818 distinct  47.973 .. 1,053.975
   AbsHoehe      100.0% filled    196,055 distinct  44.744 .. 970.906
   DachFlaech    100.0% filled    263,226 distinct  0.000 .. 167,935.751

SEMANTIC  (2)
   function      100.0% filled         88 distinct  31001_1000
   Name            2.0% filled      2,309 distinct  "Erich Mundstock Halle"

LOCATION  (4)
   AGS           100.0% filled        145 distinct  03101000
   Stadt         100.0% filled        144 distinct  Adenbüttel
   Strasse        54.0% filled     10,332 distinct  Aachener Straße
   HausNr         54.0% filled      1,786 distinct  1

DECIDE NEXT  (8)
   externRef     100.0% filled    869,327 distinct  http://repository.gdi-de.org/schemas/adv
   creationDa    100.0

  ..  read in one pass  [14.1s]

  DqDach  (2 distinct)
      1000                                          4,721,548   96.5%
      6000                                            169,795    3.5%



  creationDa  (27 distinct)
      2021-01-16                                    1,028,697   21.0%
      2020-11-23                                      934,037   19.1%
      2023-07-16                                      577,808   11.8%
      2020-11-26                                      387,109    7.9%
      2022-12-14                                      345,233    7.1%
      2021-01-26                                      334,509    6.8%
      2022-12-08                                      308,334    6.3%
      2020-11-27                                      224,051    4.6%
      2023-02-14                                      158,075    3.2%
      2020-11-29                                      110,473    2.3%
      ... and 17 more

  GrundrissA  (14 distinct)
      2017-09-29                                    1,666,115   34.1%
      2017-11-15                                    1,088,794   22.3%
      2017-08-01                                      767,174   15.7%
      2018

## 5. Classify every surface using Z

Step 02 keeps the Z ordinate, which makes the surfaces **classifiable** rather
than guessable:

| class | test |
|---|---|
| `GROUND` | flat in Z (`zmax − zmin ≤ 1 cm`) **and** sitting at the part's own `AbsHoehe` |
| `WALL` | spans Z **and** projects to ~zero XY area (vertical) |
| `ROOF` | everything else |

**Verified against CityGML's own labels**, which the shapefile export discards.
Across 20 tiles, 10,803 parts, 98,789 surfaces: every `GroundSurface` found
(100.0000 %), nothing else ever classified `GROUND` (0 false positives), and the
footprint areas identical to `+0.000000 %`. `ClosureSurface` — the virtual panes
that seal archways and carport gaps — is vertical, so all 10,591 of them land in
`WALL` and are discarded. For comparison, the original pipeline's
`groupby('gml_id')['area_m2'].idxmax()` is exact for only 99.62 % of parts,
because it picks the roof wherever a roof overhangs the walls.

Two ordering rules the cell depends on:

* **Z statistics come from the original 3D geometry**, and must be taken before
  anything is flattened.
* **Every area comes from a repaired 2D projection.** Flattening leaves 52.5 % of
  surfaces self-intersecting — a hip roof's faces meet at an apex and project
  onto shared edges — and an invalid ring yields both a wrong `.area` and a
  `TopologyException` from `union_all`. The 3D geometry is kept in place
  regardless, because the volume needs the roof faces' Z.

In [6]:
import geopandas as gpd
import numpy as np
import shapely
from shapely.geometry import Polygon

Z_TOL = 0.01   # metres. LoD2 stores mm precision, so 1 cm is generous.

# Everything the output needs, in one read. This is the expensive step - the
# geometry alone is ~4 GB - so it happens exactly once and both the deliverable
# and the experimental layer are built from it.
_READ = ['gml_id', 'externRef', 'function', 'measHeight',
         'Firsthoehe', 'Traufhoehe', 'AbsHoehe', 'DachFlaech',
         'DachNeig', 'DachOri', 'roofType', 'DachName', 'DqDach',
         'creationDa', 'GrundrissA', 'Name', 'AGS', 'Stadt', 'Strasse', 'HausNr']

print(f'Reading {len(_READ)} columns + 3D geometry from '
      f'{ALKIS_RAW_FILE.name} ({ALKIS_RAW_FILE.stat().st_size / 1e9:,.2f} GB; '
      f'~3-5 min) ...', flush=True)
t0 = time.perf_counter()
surf = gpd.read_file(ALKIS_RAW_FILE, layer='buildings', columns=_READ)
print(f'  ok  {len(surf):,} surface rows  [{time.perf_counter() - t0:,.0f}s]')

if not surf.geometry.has_z.any():
    raise AssertionError(
        'the raw layer has no Z ordinate, so surfaces cannot be classified and '
        'no exact volume is possible. Re-run step 02 with '
        'LOD2_MERGE_FLATTEN_Z = False.'
    )
print('  ok  geometry carries Z')

# alkis_id: rpartition, NEVER str.split('$$$'). pandas treats a >1-char split
# pattern as a regex, and '$$$' is three end-anchors - it matches the empty
# string at the end and every id silently becomes ''.
surf['alkis_id'] = surf['externRef'].str.rpartition('$$$')[2]
surf = surf.drop(columns='externRef')
_bad = ~surf['alkis_id'].fillna('').str.startswith('DENIAL')
if _bad.any():
    raise AssertionError(
        f'{int(_bad.sum()):,} rows carry no DENIAL-prefixed ALKIS id. '
        f'Examples: {surf.loc[_bad, "alkis_id"].head(3).tolist()}'
    )
print(f'  ok  alkis_id on every row, {surf["alkis_id"].nunique():,} distinct')

Reading 20 columns + 3D geometry from 02_alkis_lod2_raw.gpkg (5.07 GB; ~3-5 min) ...


  ok  4,891,343 surface rows  [58s]


  ok  geometry carries Z


  ok  alkis_id on every row, 869,327 distinct


In [7]:
# --- Z statistics, from the 3D geometry --------------------------------------
t0 = time.perf_counter()
geoms = surf.geometry.values
coords = shapely.get_coordinates(geoms, include_z=True)
counts = shapely.get_num_coordinates(geoms)
row_of = np.repeat(np.arange(len(surf)), counts)
zc = pd.Series(coords[:, 2])
surf['zmin'] = zc.groupby(row_of).min().values
surf['zmax'] = zc.groupby(row_of).max().values
del coords, row_of, zc

# --- 2D projection, REPAIRED, then measured ----------------------------------
g2d = shapely.force_2d(geoms)
_invalid = ~shapely.is_valid(g2d)
n_invalid = int(_invalid.sum())
if n_invalid:
    g2d[_invalid] = shapely.buffer(g2d[_invalid], 0)
print(f'  ..  repaired {n_invalid:,} of {len(surf):,} projected geometries '
      f'({100 * n_invalid / len(surf):.1f} %)')
surf['geom2d'] = g2d
surf['area_xy'] = shapely.area(g2d)

_flat      = (surf['zmax'] - surf['zmin']) <= Z_TOL
_at_ground = (surf['zmax'] - surf['AbsHoehe']).abs() <= Z_TOL
_vertical  = surf['area_xy'] < 0.01
surf['surface_class'] = np.where(_flat & _at_ground, 'GROUND',
                          np.where((~_flat) & _vertical, 'WALL', 'ROOF'))
print(f'  ok  classified  [{time.perf_counter() - t0:,.0f}s]')
print('  ..  ' + str(surf['surface_class'].value_counts().to_dict()))

# Exactly one GROUND per part is what the whole approach rests on, so it is
# measured on the full region rather than trusted from the tile sweeps.
n_surfaces = surf.groupby('gml_id').size()
per_part = surf[surf['surface_class'] == 'GROUND'].groupby('gml_id').size()
n_parts = len(n_surfaces)
print()
print(f'  ..  parts total          : {n_parts:,}')
print(f'  ..  parts with 1 GROUND  : {int((per_part == 1).sum()):,} '
      f'({100 * (per_part == 1).sum() / n_parts:.3f} %)')
print(f'  ..  parts with >1 GROUND : {int((per_part > 1).sum()):,}')
print(f'  ..  parts with NO GROUND : {n_parts - len(per_part):,}')

_bad_parts = set(n_surfaces.index) - set(per_part[per_part == 1].index)
if _bad_parts:
    _sub = surf[surf['gml_id'].isin(_bad_parts)]
    print(f'  !!  {len(_bad_parts):,} parts cannot yield a footprint and are '
          f'excluded from the output:')
    print(f'        surface rows each : '
          f'{_sub.groupby("gml_id").size().value_counts().sort_index().to_dict()}')
    print(f'        classes present   : {_sub["surface_class"].value_counts().to_dict()}')
    # A part that cannot yield a footprint takes its whole ALKIS object with it
    # when no other part of that object survives. The sliver filter in 6b
    # asserts 'no building lost' relative to THIS point, so the loss is
    # reported here or nowhere. Measured: 11 objects, 9 of them masts/towers.
    _lost_obj = (set(surf.loc[surf['gml_id'].isin(_bad_parts), 'alkis_id'])
                 - set(surf.loc[~surf['gml_id'].isin(_bad_parts), 'alkis_id']))
    print(f'  !!  {len(_lost_obj):,} ALKIS objects have no other part and drop out of '
          f'both outputs: {sorted(_lost_obj)}')

  ..  repaired 2,567,001 of 4,891,343 projected geometries (52.5 %)


  ok  classified  [94s]


  ..  {'WALL': 2095872, 'ROOF': 1410184, 'GROUND': 1385287}



  ..  parts total          : 1,385,279
  ..  parts with 1 GROUND  : 1,385,265 (99.999 %)
  ..  parts with >1 GROUND : 11
  ..  parts with NO GROUND : 3


  !!  14 parts cannot yield a footprint and are excluded from the output:
        surface rows each : {1: 2, 2: 2, 3: 10}
        classes present   : {'GROUND': 22, 'ROOF': 10, 'WALL': 4}


  !!  11 ALKIS objects have no other part and drop out of both outputs: ['DENIAL01000031JG', 'DENIAL010000cUAn', 'DENIAL010000cl9h', 'DENIAL03000064TQ', 'DENIAL610000789T', 'DENIAL610000dM9C', 'DENIAL610000e2Ry', 'DENIAL610000fAmZ', 'DENIAL610000gvKV', 'DENIAL8400002Qof', 'DENIAL8400005VLK']


## 6. Build the part layer — footprint and both volumes

The geometry becomes the **`GROUND` polygon**, flat, in EPSG:25832. Everything
3D is retained as numbers.

### The two volumes

| column | how it is computed |
|---|---|
| `volume_3d_m3` | Σ over `ROOF` faces, each **clipped to the part's footprint**, of the exact prism between the ground plane and the face |
| `volume_old_m3` | `area_m2 × height_top_m` — the original pipeline's method |
| `volume_ratio` | `volume_old_m3 / volume_3d_m3` |

`volume_3d_m3` is exact for a flat ground and planar roof faces, which is what
LoD2 guarantees. Think of the building as a bundle of vertical columns standing
on the footprint: each roof face contributes its own shadow area times its
average height above the ground. No per-roof-shape formula is needed — gable,
hip, pyramid and barrel all fall out of the same sum.

**The average height is not the mean of the corner heights.** For a plane the
average over an *area* is the height at the **area centroid**, whereas averaging
the corner coordinates gives the **vertex centroid** — and those coincide only
for triangles and parallelograms. An earlier version of this cell took the
vertex mean; measured against the exact integral over 2,201 buildings it was
wrong for **25.1 %** of them, −1.28 % in aggregate but individual buildings off
by up to **6.7×**, concentrated in faces with five or more unevenly spaced
corners. Per-building volume *is* the redistribution weight, so the cell now
fan-triangulates each face with signed 2D areas, which is exact for any planar
polygon, convex or not.

**A face is clipped to its own footprint.** LGLN attaches a hip-type roof face
to *one* part of a multi-part building while the face spans the whole roof, so
a per-part prism sum counts the volume under it over the neighbouring parts as
well. On the unclipped run 17,469 parts showed a volume *above* `area × ridge
height`, which no roof can produce. Recomputing the unclipped volume of every part the
clip touched puts the error at 19,020 parts (1.4 %) in 16,704 buildings,
overstated by 53 % on those parts and by 4.47 M m³, 0.66 % of the region, in
total; per building the overstatement was a median 13 %, 2,701 buildings were
above 20 % and 12 had their volume more than doubled. A face whose
projection leaves the footprint is intersected with the ground polygon and its
plane integrated over the clipped pieces (exact for a planar face); every other
face keeps the fan triangulation, so their volumes are unchanged. Whether a
face *leaves* the footprint is decided by width, not area: LoD2 coordinates sit
on a 1 mm grid, so a face along a shared edge can stick out by a millimetre-wide
sliver, and such a sliver is shaved away by `ALKIS_ROOF_SPILL_MIN_WIDTH_M`
before it can count (a real spill is at least 15 cm wide here). Checked against
a 0.25 m grid integral over the dissolved footprint of 1,984 sample buildings,
the clipped sum is within 0.05 % overall; no multi-part building differs by
more than 5 %, and the only four buildings above 5 % (max 6 %) are single-part
flat roofs under 10 m² where the grid itself is coarse. `n_roof_clipped`
records how many faces the clip touched. A closed-mesh volume is *not* an
alternative: the part solids are not closed (walls shared between parts are
absent), so the divergence theorem returns garbage on them.

The integral runs vectorised over all two million faces at once (numpy over
shapely's coordinate arrays), which took the cell from five and a half
minutes to about one. The readable per-face implementation stays in the cell
and is run on every 500th roof row each time as a check; a disagreement fails
the cell rather than producing a volume nobody has verified.

`volume_old_m3` treats the building as a box up to the **ridge**, so it counts
the empty wedge between eaves and ridge as solid:

```
        ╱▔╲
      ╱▒▒▒▒▒╲        ▒ = air that volume_old_m3 counts as building
    ╱─────────╲
    │         │
    └─────────┘
```

Both are kept so the difference against the original pipeline stays auditable,
and `volume_ratio` makes it inspectable per building. The region-wide
overstatement is printed by the cell below — it is exactly 1.000 for flat roofs,
where no wedge exists, and rises with roof pitch.

### Column names

Two prefixes carry the whole convention:

```
height_*   metres ABOVE THE GROUND at this building
elev_*     metres ABOVE SEA LEVEL
top        the ridge - the highest line of the roof
eaves      the MEAN eaves height - LGLN's 3D-Kundentag 2019 slides list the field as *Mittlere Traufhöhe*
```

```
        ╱▔▔╲   <- top   (ridge)
      ╱      ╲
    ╱──────────╲ <- eaves (the attribute is the MEAN eaves height)
    │          │
    └──────────┘ <- ground
```

So `height_top_m` is ground → highest point, `height_eaves_m` is ground → mean
eaves, and `elev_ground_m` / `elev_top_m` are the same points measured from
sea level. Because the eaves value is a *mean*, a lean-to roof's low edge sits
below it (about 0.8 m on median across the region, 1.25 m in the anomalous parts inspected), so `area × height_eaves_m` is not a lower
bound on the volume - 10,356 lean-to parts fall under it legitimately. That distinction is not cosmetic: this region spans 45–970 m of
terrain, so mixing an elevation into a height calculation is a large error, and
the AdV names (`Firsthoehe`, `Traufhoehe`, `AbsHoehe`) give no hint which is
which.

One column keeps a misleading name deliberately: **`roof_pitch_deg` is measured
from the vertical**, so a flat roof reads `90.0`, not `0`. Renaming would not
stop that surprising anyone, so it is documented instead.

Three more attributes read differently once the LGLN and AdV documents are
consulted, and the columns follow the documents:

* **`roof_code` and `roof_shape` are two classifications**, the AdV 15-code
  list and LGLN's 24-name list, not a code and its label. Code `5000`
  (Mischform) maps to four names, `3500` to two, `9999` to nine.
* **`roof_source`** (`DqDach`) is the *Datenquelle Dachhöhe*: `1000` laser
  scan, `6000` MANUELL. The 6000 rows are the manually post-processed
  landmarks and "ortsprägende Gebäude", the *better*-checked subset - 88 % of
  fire stations and 86 % of chapels have a manually processed part against
  3.8 % of residential buildings - not estimated
  roofs, which is what the column used to be called.
* **`roof_is_fallback`** marks a flat roof with code `9999`: per the AdV format
  description that is a building modelled from a LoD1 object; LGLN's 2019 slides
  name the cause, a failed roof recognition. 54,060 parts. Their volume is a box at laser height, so
  it carries the same overstatement `volume_old_m3` has.

Two facts about the source that bound what any volume here can mean: the
ground surface sits at the *lowest* terrain intersection and heights are good
to about 1 m (AdV, SIG3D), so on sloping ground the solid includes a wedge
below the uphill terrain; and roof overhangs, dormers and chimneys are not
modelled at all (LGLN), which is why roof faces tile a footprint exactly.


In [8]:
def _fan_volume(poly, ground_z):
    # Volume between the ground plane and ONE polygon face, EXACTLY.
    #
    # The volume under a planar face is  INT z dA - ground_z * area, and for a
    # plane INT z dA equals the area times z at the face's AREA centroid.
    #
    # An earlier version used `area x mean(vertex z)` - the VERTEX centroid -
    # which coincides with the area centroid only for triangles and
    # parallelograms. Measured over 2,201 buildings that shortcut was wrong for
    # 25.1 % of them: -1.28 % in total, but individual buildings off by up to
    # 6.7x, concentrated in faces with 5+ unevenly spaced corners. Per-building
    # volume IS the redistribution weight, so that was not acceptable.
    #
    # Fan triangulation with SIGNED 2D areas is exact for any planar face,
    # convex or not: z at a triangle's centroid is the plain mean of its three
    # vertex heights, and the signed areas make concave parts cancel correctly.
    tot = 0.0
    for ring, sign in ([(poly.exterior, 1.0)] +
                       [(h, -1.0) for h in poly.interiors]):
        c = np.asarray(ring.coords)
        if len(c) > 1 and np.allclose(c[0], c[-1]):
            c = c[:-1]
        if len(c) < 3:
            continue
        x, y, z = c[:, 0], c[:, 1], c[:, 2]
        # signed area of each fan triangle (v0, vi, vi+1)
        sa = 0.5 * ((x[1:-1] - x[0]) * (y[2:] - y[0]) -
                    (x[2:] - x[0]) * (y[1:-1] - y[0]))
        zbar = (z[0] + z[1:-1] + z[2:]) / 3.0
        contrib = float(np.sum(sa * (zbar - ground_z)))
        # Normalise THIS RING's winding, not the running total. The rings of
        # one MultiPolygon are not consistently wound, so summing raw signed
        # contributions makes faces cancel: two identical flat faces with
        # opposite winding gave 0 instead of 800. Deciding the sign per ring
        # keeps the fan cancellation that makes concave faces exact, while
        # every face still adds.
        if float(np.sum(sa)) < 0:
            contrib = -contrib
        tot += sign * contrib
    return tot


def _face_plane(poly):
    """z = a x + b y + c through the face: Newell normal, passing through the
    vertex mean. None for a vertical face."""
    p = np.asarray(poly.exterior.coords)[:-1]
    if len(p) < 3:
        return None
    q = np.roll(p, -1, axis=0)
    nx = np.sum((p[:, 1] - q[:, 1]) * (p[:, 2] + q[:, 2]))
    ny = np.sum((p[:, 2] - q[:, 2]) * (p[:, 0] + q[:, 0]))
    nz = np.sum((p[:, 0] - q[:, 0]) * (p[:, 1] + q[:, 1]))
    if abs(nz) < 1e-9:
        return None
    a, b = -nx / nz, -ny / nz
    cx, cy, cz = p.mean(axis=0)
    return a, b, cz - a * cx - b * cy


def _plane_volume(poly2d, plane, ground_z):
    """Volume under a plane over a 2D polygon, exactly: area x height at the
    AREA centroid, which shapely computes with holes accounted for."""
    a, b, c = plane
    tot = 0.0
    for g in getattr(poly2d, 'geoms', [poly2d]):
        if g.is_empty or g.area == 0.0:
            continue
        cen = g.centroid
        tot += g.area * (a * cen.x + b * cen.y + c - ground_z)
    return tot


def _prism_volume(geom, ground_z, footprint):
    """Volume between the ground plane and this roof row's faces, the number of
    faces integrated, and the number that had to be clipped to the footprint.

    A raw row is a MultiPolygon that may hold several faces (both sides of a
    gable, for instance) and, now and then, a vertical polygon that belongs to a
    wall; every polygon is treated on its own, and the vertical ones, whose
    projection is empty, are skipped and not counted.

    THE CLIP. LGLN attaches a hip-type roof face to ONE part of a multi-part
    building while the face spans the whole roof, so its prism over the
    neighbouring parts would be counted here AND under those parts. A face whose
    projection leaves this part's footprint is intersected with the ground
    polygon and its plane integrated over the clipped pieces - exact for a planar
    face. Whether it LEAVES the footprint is a question of width, not area: LoD2
    coordinates sit on a 1 mm grid, so a face along a shared edge can stick out
    by a millimetre-wide sliver; the spill is shaved by half
    ALKIS_ROOF_SPILL_MIN_WIDTH_M on every side and only what survives counts (a
    real spill is at least 15 cm wide here). Everything not clipped keeps the fan
    triangulation, so the volume of an ordinary part is bit-identical to the
    unclipped computation; a face without a recoverable plane (a twisted ring)
    keeps it too. Before the clip 17,469 parts had a volume above `area x ridge
    height`, which no roof can produce; see config.py.
    """
    tot = 0.0
    n_faces = 0
    n_clip = 0
    for poly in getattr(geom, 'geoms', [geom]):
        p2 = shapely.force_2d(poly)
        if not p2.is_valid:
            p2 = p2.buffer(0)
        if p2.is_empty:
            continue
        n_faces += 1
        if not shapely.covers(footprint, p2):
            spill = p2.difference(footprint)
            if not spill.is_empty and not spill.buffer(-ALKIS_ROOF_SPILL_MIN_WIDTH_M / 2).is_empty:
                plane = _face_plane(poly)
                if plane is not None:
                    tot += _plane_volume(p2.intersection(footprint), plane, ground_z)
                    n_clip += 1
                    continue
        tot += _fan_volume(poly, ground_z)
    return tot, n_faces, n_clip


# --- the same integral, vectorised --------------------------------------------
# _prism_volume above is the readable reference: one Python call per face, and
# at two million faces the interpreter overhead alone took five minutes. The
# three functions below do the identical arithmetic over shapely's coordinate
# arrays with numpy, ~75x fewer Python-level calls, and are checked against the
# reference on a sample of rows every run (see below the call).

def _fan_vectorised(faces, gz_face):
    """_fan_volume for an array of polygon faces at once: exact fan triangulation
    per ring, holes subtracted, winding normalised per ring."""
    if len(faces) == 0:
        return np.zeros(0)
    rings, poly_of_ring = shapely.get_rings(faces, return_index=True)
    is_ext = np.r_[True, poly_of_ring[1:] != poly_of_ring[:-1]]          # first ring of each polygon
    ring_sign = np.where(is_ext, 1.0, -1.0)
    xyz, ring_of = shapely.get_coordinates(rings, include_z=True, return_index=True)
    x, y, z = xyz[:, 0], xyz[:, 1], xyz[:, 2]
    starts = np.flatnonzero(np.r_[True, ring_of[1:] != ring_of[:-1]])
    x0, y0, z0 = x[starts[ring_of]], y[starts[ring_of]], z[starts[ring_of]]
    g = gz_face[poly_of_ring][ring_of]
    # triangle (v0, v_j, v_j+1) for every consecutive pair inside one ring with j
    # not the start vertex. The closing vertex repeats v0, so its triangle has
    # exactly zero area and adds nothing.
    j = np.arange(len(x) - 1)
    ok = (ring_of[j] == ring_of[j + 1]) & (j != starts[ring_of[j]])
    j = j[ok]
    sa = 0.5 * ((x[j] - x0[j]) * (y[j + 1] - y0[j]) - (x[j + 1] - x0[j]) * (y[j] - y0[j]))
    zbar = (z0[j] + z[j] + z[j + 1]) / 3.0
    n_rings = len(rings)
    contrib = np.bincount(ring_of[j], weights=sa * (zbar - g[j]), minlength=n_rings)
    area_s = np.bincount(ring_of[j], weights=sa, minlength=n_rings)
    contrib = np.where(area_s < 0, -contrib, contrib) * ring_sign
    return np.bincount(poly_of_ring, weights=contrib, minlength=len(faces))


def _planes(faces):
    """_face_plane for an array of faces: Newell normal of the exterior ring,
    plane through the vertex mean (closing vertex excluded). ok=False if vertical."""
    ext = shapely.get_exterior_ring(faces)
    xyz, ring_of = shapely.get_coordinates(ext, include_z=True, return_index=True)
    x, y, z = xyz[:, 0], xyz[:, 1], xyz[:, 2]
    n = len(faces)
    i = np.arange(len(x) - 1)
    i = i[ring_of[i] == ring_of[i + 1]]
    nx = np.bincount(ring_of[i], weights=(y[i] - y[i + 1]) * (z[i] + z[i + 1]), minlength=n)
    ny = np.bincount(ring_of[i], weights=(z[i] - z[i + 1]) * (x[i] + x[i + 1]), minlength=n)
    nz = np.bincount(ring_of[i], weights=(x[i] - x[i + 1]) * (y[i] + y[i + 1]), minlength=n)
    last = np.r_[ring_of[1:] != ring_of[:-1], True]
    cnt = np.bincount(ring_of[~last], minlength=n).astype(float)
    cx = np.bincount(ring_of[~last], weights=x[~last], minlength=n) / cnt
    cy = np.bincount(ring_of[~last], weights=y[~last], minlength=n) / cnt
    cz = np.bincount(ring_of[~last], weights=z[~last], minlength=n) / cnt
    ok = np.abs(nz) >= 1e-9
    nzs = np.where(ok, nz, 1.0)
    a, b = -nx / nzs, -ny / nzs
    return a, b, cz - a * cx - b * cy, ok


def roof_volumes(geoms, gz, fps):
    """_prism_volume for all roof rows at once. Returns per ROW: the volume, the
    number of faces integrated, the number of faces clipped to the footprint."""
    n_rows = len(geoms)
    faces, row_of = shapely.get_parts(np.asarray(geoms, dtype=object), return_index=True)
    f2 = shapely.force_2d(faces)
    inv = ~shapely.is_valid(f2)
    if inv.any():
        f2[inv] = shapely.buffer(f2[inv], 0)
    keep = ~shapely.is_empty(f2)          # a vertical polygon inside a ROOF row projects to nothing
    faces, f2, row_of = faces[keep], f2[keep], row_of[keep]
    fp = np.asarray(fps, dtype=object)[row_of]
    g0 = np.asarray(gz, dtype=float)[row_of]
    clip = np.zeros(len(faces), dtype=bool)
    idx = np.flatnonzero(~shapely.covers(fp, f2))
    if len(idx):
        spill = shapely.difference(f2[idx], fp[idx])
        real = ~shapely.is_empty(shapely.buffer(spill, -ALKIS_ROOF_SPILL_MIN_WIDTH_M / 2))
        clip[idx[real]] = True
    if clip.any():
        cidx = np.flatnonzero(clip)
        a, b, c, ok = _planes(faces[cidx])
        clip[cidx[~ok]] = False           # no recoverable plane: keep the fan integral
        cidx, a, b, c = cidx[ok], a[ok], b[ok], c[ok]
    vol = np.zeros(len(faces))
    fan = np.flatnonzero(~clip)
    vol[fan] = _fan_vectorised(faces[fan], g0[fan])
    if clip.any():
        pieces, piece_of = shapely.get_parts(shapely.intersection(f2[cidx], fp[cidx]), return_index=True)
        area = shapely.area(pieces)
        # A clipped face can leave zero-area slivers (lines, points) in the
        # result. They carry no volume and a degenerate one has an EMPTY centroid,
        # on which GEOS raises, so they go before the centroid is asked for -
        # exactly the pieces _plane_volume skips.
        has_area = area > 0
        pieces, piece_of, area = pieces[has_area], piece_of[has_area], area[has_area]
        cen = shapely.centroid(pieces)
        zc = (a[piece_of] * shapely.get_x(cen) + b[piece_of] * shapely.get_y(cen) + c[piece_of]
              - g0[cidx][piece_of])
        vol[cidx] = np.bincount(piece_of, weights=area * zc, minlength=len(cidx))
    return (np.bincount(row_of, weights=vol, minlength=n_rows),
            np.bincount(row_of, minlength=n_rows),
            np.bincount(row_of[clip], minlength=n_rows))


usable = per_part[per_part == 1].index
ground = (surf[(surf['surface_class'] == 'GROUND') & surf['gml_id'].isin(usable)]
          .set_index('gml_id'))
print(f'  ..  parts with a usable GROUND footprint: {len(ground):,}')
# Prepared once, so the covers() test per face is a cheap index lookup.
shapely.prepare(ground['geom2d'].values)

roof = surf[(surf['surface_class'] == 'ROOF') &
            surf['gml_id'].isin(ground.index)].copy()
roof['_gz'] = roof['gml_id'].map(ground['zmax'])
roof['_fp'] = roof['gml_id'].map(ground['geom2d'])
print(f'  ..  computing volume_3d_m3 over {len(roof):,} roof rows '
      f'(about a minute) ...', flush=True)
t0 = time.perf_counter()
_v, _nf, _nc = roof_volumes(roof.geometry.values, roof['_gz'].values, roof['_fp'].values)
roof['_v'] = _v
# faces, not rows: a gable's two faces usually share one MultiPolygon row - and
# only the polygons that were integrated count, a vertical one in a ROOF row not
roof['_nfaces'] = _nf
roof['_nclip'] = _nc
# The vectorised code is held to the readable per-face reference on every run:
# every 500th roof row must agree to a microlitre and in both counts.
_chk = roof.iloc[::500]
_ref = [_prism_volume(g, gz, fp)
        for g, gz, fp in zip(_chk.geometry.values, _chk['_gz'].values, _chk['_fp'].values)]
_dv = float(np.abs(_chk['_v'].values - np.array([r[0] for r in _ref])).max())
if (_dv > 1e-6
        or (_chk['_nfaces'].values != np.array([r[1] for r in _ref])).any()
        or (_chk['_nclip'].values != np.array([r[2] for r in _ref])).any()):
    raise AssertionError(
        f'the vectorised volume disagrees with the per-face reference on the check '
        f'rows (max |dV| = {_dv:.3e} m3). Do not trust volume_3d_m3 from this run.'
    )
print(f'  ok  vectorised volume agrees with the per-face reference on {len(_chk):,} '
      f'check rows (max |dV| {_dv:.1e} m3)')
vol3d = roof.groupby('gml_id')['_v'].sum()
n_roof_faces = roof.groupby('gml_id')['_nfaces'].sum()
n_roof_clipped = roof.groupby('gml_id')['_nclip'].sum()
print(f'  ok  [{time.perf_counter() - t0:,.0f}s]  '
      f'{int(roof["_nfaces"].sum()):,} faces in {len(roof):,} rows; '
      f'{int(roof["_nclip"].sum()):,} faces clipped to their footprint in '
      f'{int((n_roof_clipped > 0).sum()):,} parts')

# --- assemble -----------------------------------------------------------------
bld = gpd.GeoDataFrame(
    ground.drop(columns=['zmin', 'zmax', 'area_xy', 'surface_class', 'geom2d',
                         ground.geometry.name]),
    geometry=ground['geom2d'].values, crs=ground.crs)
bld['area_m2'] = ground['area_xy'].values

bld = bld.rename(columns=ALKIS_RENAME)
bld['height_eaves_m'] = (bld['elev_eaves_m'] - bld['elev_ground_m']).round(3)
bld['volume_3d_m3']   = vol3d.reindex(bld.index).fillna(0.0).round(1)
bld['volume_old_m3']  = (bld['area_m2'] * bld['height_top_m']).round(1)
bld['volume_ratio']   = (bld['volume_old_m3'] /
                         bld['volume_3d_m3'].replace(0, np.nan)).round(4)
bld['n_surfaces']     = n_surfaces.reindex(bld.index).astype('int32')
bld['n_roof_faces']   = n_roof_faces.reindex(bld.index).fillna(0).astype('int32')
bld['n_roof_clipped'] = n_roof_clipped.reindex(bld.index).fillna(0).astype('int32')
# the LoD1 fallback: roof recognition failed, the building is a flat box
bld['roof_is_fallback'] = ((bld['roof_code'] == ALKIS_ROOF_FALLBACK['roof_code'])
                           & (bld['roof_shape'] == ALKIS_ROOF_FALLBACK['roof_shape']))
bld['area_m2']        = bld['area_m2'].round(2)
bld = bld.reset_index().rename(columns={'index': 'gml_id'})
if 'gml_id' not in bld.columns:
    bld = bld.rename(columns={bld.columns[0]: 'gml_id'})

# The ALKIS_OUTPUT_COLS selection happens in section 6b, after the sliver
# filter, so that the dropped-row reporting can still see every column.
require_unique(bld, 'gml_id', 'alkis buildings')
print()
print('  ..  the volume comparison, region-wide:')
print(f'        volume_old_m3 total : {bld["volume_old_m3"].sum() / 1e6:>10,.1f} million m3')
print(f'        volume_3d_m3  total : {bld["volume_3d_m3"].sum() / 1e6:>10,.1f} million m3')
print(f'        old overstates by   : '
      f'{100 * (bld["volume_old_m3"].sum() / bld["volume_3d_m3"].sum() - 1):+.2f} %')
print(f'        volume_ratio median / mean / p95: '
      f'{bld["volume_ratio"].median():.3f} / {bld["volume_ratio"].mean():.3f} / '
      f'{bld["volume_ratio"].quantile(.95):.3f}')
# No roof can hold more than a box up to its own ridge. Before the clip 17,469
# parts did; after it this should be ~0, and anything left is a face that
# overlaps another face INSIDE the footprint (a cross-gable modelled as two
# roofs passing through each other), which the clip does not address.
_over = bld['volume_3d_m3'] > bld['area_m2'] * bld['height_top_m'] * 1.01 + 0.5
print(f'        parts with volume_3d_m3 above area x ridge height: {int(_over.sum()):,} '
      f'(excess {(bld.loc[_over, "volume_3d_m3"] - bld.loc[_over, "area_m2"] * bld.loc[_over, "height_top_m"]).sum() / 1e6:,.3f} M m3)')
print(f'        parts with a clipped face : {int((bld["n_roof_clipped"] > 0).sum()):,}   '
      f'LoD1 flat-box fallbacks : {int(bld["roof_is_fallback"].sum()):,}')
print()
print('  ..  by roof shape (median volume_ratio):')
_t = bld.groupby('roof_shape').agg(n=('volume_ratio', 'size'),
                                   ratio=('volume_ratio', 'median'))
print(_t.sort_values('n', ascending=False).head(10).round(3).to_string())


  ..  parts with a usable GROUND footprint: 1,385,265


  ..  computing volume_3d_m3 over 1,410,174 roof rows (about a minute) ...


  ok  vectorised volume agrees with the per-face reference on 2,821 check rows (max |dV| 3.6e-12 m3)


  ok  [65s]  2,028,425 faces in 1,410,174 rows; 60,146 faces clipped to their footprint in 19,110 parts


  ok  alkis buildings.gml_id: unique and non-null (1,385,265)

  ..  the volume comparison, region-wide:
        volume_old_m3 total :      773.1 million m3
        volume_3d_m3  total :      677.4 million m3
        old overstates by   : +14.13 %
        volume_ratio median / mean / p95: 1.000 / 1.089 / 1.346
        parts with volume_3d_m3 above area x ridge height: 4 (excess 0.000 M m3)
        parts with a clipped face : 19,110   LoD1 flat-box fallbacks : 74,511

  ..  by roof shape (median volume_ratio):


                          n  ratio
roof_shape                        
PolyFlatRoof         863431  1.000
GableRoof            367615  1.237
LeanToRoof            65741  1.171
HipAndGableRoof       40644  1.211
HipRoof               26164  1.260
GableAndHippedGRoof    8424  1.304
HippedGableRoof        5129  1.320
PyramidRoof            3340  1.219
CutGableRoof           1514  1.047
TentRoof               1463  1.293


## 6b. Drop sliver parts

54,093 parts (3.90 %) have a footprint under 1 m². They are **correctly**
classified ground surfaces — flat, and sitting at `AbsHoehe`. They simply are
not structures: the bounding rectangle is a median **1.48 m × 0.25 m**, aspect
ratio 4.81, and 31 % are thinner than 1:10. That is a wall-thickness leftover,
the gap where two adjoining outlines fail to meet, or a facade jog that was
given its own part. The 22 % that are compact (~0.5 × 0.5 m, 3.5 m tall) look
like chimneys, vents and stair heads modelled separately.

They are also where the geometry is worst: **33,722 of the 66,563 parts whose
volume falls outside its own plausible bracket are these slivers**, so removing
them takes out about half the defective population.

The trade:

```
rows removed              53,236     3.84 % of parts (857 of the 54,093 kept as their building's only part)
volume lost              118,103 m3  0.0174 % of the region
buildings lost                    0  (see below)
```

853 of these slivers **are** their whole building — their `alkis_id` has no
other part — so dropping everything below the threshold would delete 857 ALKIS
objects from the region. `ALKIS_KEEP_LARGEST_PART_PER_BUILDING` keeps the
largest part of any building that would otherwise vanish, so the filter removes
geometry noise without ever removing a building. The cell asserts that.

Note this is a **size** decision, deliberately separate from the surface
classification: a sliver's floor is perfectly flat and perfectly at ground
level, so no classification rule could catch it. 1 m² is a conservative line
where the geometry is unambiguously an artifact — the larger question of whether
a 28 m² `31001_2000` shed counts as a building is a different one, still open.

In [9]:
n_before = len(bld)
print(f'{n_before:,} parts in')

tiny = bld['area_m2'] < ALKIS_MIN_PART_AREA_M2
# A building is "at risk" when every one of its parts is below the threshold.
biggest_part = bld.groupby('alkis_id')['area_m2'].transform('max')
at_risk = biggest_part < ALKIS_MIN_PART_AREA_M2
if ALKIS_KEEP_LARGEST_PART_PER_BUILDING:
    # idxmax over a RangeIndex gives the one row per at-risk building to keep.
    keep_idx = bld.loc[at_risk].groupby('alkis_id')['area_m2'].idxmax()
    rescued = bld.index.isin(keep_idx)
else:
    rescued = np.zeros(len(bld), dtype=bool)
drop = tiny & ~rescued

print(f'  ..  footprint < {ALKIS_MIN_PART_AREA_M2:g} m2                    : {int(tiny.sum()):>8,}')
print(f'  ..  rescued (their building has no bigger part) : {int((tiny & rescued).sum()):>8,}')
print(f'  ..  dropped                                     : {int(drop.sum()):>8,}')
_vd = bld.loc[drop, 'volume_3d_m3'].sum()
print(f'  ..  volume dropped : {_vd:,.1f} m3 '
      f'({100 * _vd / bld["volume_3d_m3"].sum():.4f} % of the region)')

_before_ids = set(bld['alkis_id'])
bld = bld.loc[~drop].reset_index(drop=True)
_lost = _before_ids - set(bld['alkis_id'])
if _lost:
    raise AssertionError(
        f'{len(_lost):,} ALKIS buildings disappeared with the sliver filter, '
        f'which the rescue rule exists to prevent. Examples: {sorted(_lost)[:3]}'
    )
print(f'  ok  {len(bld):,} parts kept, {bld["alkis_id"].nunique():,} buildings '
      f'(none lost)')
print(f'  ..  smallest footprint kept: {bld["area_m2"].min():.4f} m2 '
      f'(a rescued single-part building)')

# --- now select and check the output columns --------------------------------
missing = [c for c in ALKIS_OUTPUT_COLS if c not in bld.columns]
if missing:
    raise AssertionError(f'ALKIS_OUTPUT_COLS names columns that do not exist: {missing}')
extra = [c for c in bld.columns if c not in ALKIS_OUTPUT_COLS]
if extra:
    print(f'  ..  dropping columns not in ALKIS_OUTPUT_COLS: {extra}')
bld = bld[ALKIS_OUTPUT_COLS]
require_unique(bld, 'gml_id', 'alkis buildings')
assert_no_empty_columns(bld, 'alkis buildings')
print(f'  ok  {len(bld):,} rows x {len(bld.columns)} columns '
      f'({len(bld) - n_before:+,} vs before this section)')

1,385,265 parts in


  ..  footprint < 1 m2                    :   54,093
  ..  rescued (their building has no bigger part) :      857
  ..  dropped                                     :   53,236
  ..  volume dropped : 118,102.6 m3 (0.0174 % of the region)


  ok  1,332,029 parts kept, 869,316 buildings (none lost)
  ..  smallest footprint kept: 0.0400 m2 (a rescued single-part building)


  ok  alkis buildings.gml_id: unique and non-null (1,332,029)


  ok  alkis buildings: 30 columns, none empty
  ok  1,332,029 rows x 30 columns (-53,236 vs before this section)


## 7. Write the deliverable

GeoPackage, not shapefile: one file, no 2 GB component cap, no 10-character
field-name truncation, and proper NULLs. The file is unlinked first because a
GeoPackage write *appends*, so a stale layer from an earlier run would otherwise
survive beside the new one.

If this fails with `[WinError 32] being used by another process`, something has
the file open — QGIS, or another notebook's kernel. Note that an open
**read-only** SQLite handle is enough to block deletion on Windows while the file
still reports as writable.

In [10]:
if ALKIS_BUILDINGS_FILE.exists():
    try:
        ALKIS_BUILDINGS_FILE.unlink()
    except PermissionError as e:
        raise RuntimeError(
            f'{ALKIS_BUILDINGS_FILE.name} is locked by another process, so it '
            'cannot be replaced. QGIS holds a GeoPackage open while the layer is '
            'loaded, and so does another notebook kernel with an open sqlite3 '
            'connection - even a read-only one. Close them and rerun this cell. '
            f'Original error: {e}'
        ) from None

print(f'Writing {len(bld):,} rows x {len(bld.columns)} columns to '
      f'{ALKIS_BUILDINGS_FILE.name} ...', flush=True)
t0 = time.perf_counter()
bld.to_file(ALKIS_BUILDINGS_FILE, layer='buildings', driver='GPKG')
print(f'  ok  {ALKIS_BUILDINGS_FILE.stat().st_size / 1e6:,.1f} MB  '
      f'[{time.perf_counter() - t0:,.0f}s]')

check = gpd.read_file(ALKIS_BUILDINGS_FILE, layer='buildings', rows=5)
print(f'  ok  read back: CRS {check.crs}, '
      f'geometry {check.geometry.geom_type.unique().tolist()}, 3D {check.geometry.has_z.any()}')
print()
print(check.drop(columns='geometry').head(5).to_string())
print()
print('  ..  columns:')
for c in bld.columns:
    if c == 'geometry':
        continue
    nn = bld[c].notna().sum()
    print(f'        {c:<18} {100 * nn / len(bld):>6.1f} % filled   {str(bld[c].dtype):<10}')

Writing 1,332,029 rows x 30 columns to 03_alkis_buildings.gpkg ...


  ok  661.2 MB  [19s]
  ok  read back: CRS EPSG:25832, geometry ['MultiPolygon'], 3D False

             gml_id          alkis_id  area_m2  volume_3d_m3  volume_old_m3  volume_ratio  height_top_m  height_eaves_m  elev_ground_m  elev_eaves_m  elev_top_m    function    roof_shape  roof_code  roof_is_fallback  roof_area_m2  roof_pitch_deg  roof_azimuth_deg  name       ags       city    street house_number  n_surfaces  n_roof_faces  n_roof_clipped  roof_source  created_on plan_acquired_on
0  DENILD61000068Eu  DENIAL61000068Eu   186.91         927.0         1130.6        1.2196         6.049           3.870         61.105        64.975      67.154  31001_1000     GableRoof       3100             False       198.042          70.702           -28.196  None  03157001  Edemissen  Grashöfe           14           3             2               0         1000  2020-11-23       2017-09-29
1  DENILD61000068Es  DENIAL61000068Es    75.55         438.5          606.8        1.3838         8.032         

        roof_shape          100.0 % filled   str       
        roof_code           100.0 % filled   int64     
        roof_is_fallback    100.0 % filled   bool      
        roof_area_m2        100.0 % filled   float64   
        roof_pitch_deg      100.0 % filled   float64   
        roof_azimuth_deg    100.0 % filled   float64   
        name                  1.8 % filled   str       
        ags                 100.0 % filled   str       
        city                100.0 % filled   str       
        street               49.4 % filled   str       


        house_number         49.4 % filled   str       
        n_surfaces          100.0 % filled   int32     
        n_roof_faces        100.0 % filled   int32     
        n_roof_clipped      100.0 % filled   int32     
        roof_source         100.0 % filled   int64     
        created_on          100.0 % filled   str       
        plan_acquired_on     98.8 % filled   str       


## 8. Aggregate to real ALKIS buildings

Same data, collapsed from 1,332,029 parts to **869,316 ALKIS objects**. Written
separately rather than instead — the part layer stays authoritative because this
step is lossy in ways worth naming.

### How each column is collapsed

| how | columns |
|---|---|
| **summed** | `volume_3d_m3`, `volume_old_m3`, `roof_area_m2`, `n_surfaces`, `n_roof_faces`, `n_roof_clipped` — additive, safe |
| **dissolved** | `geometry`, and `area_m2` recomputed *from the dissolved shape* rather than summed, so overlapping parts are not double counted |
| **largest part** | `function`, `roof_shape`, `roof_code`, address — taken from the part with the biggest ground area |
| **extremes** | `height_top_max_m`, `height_eaves_max_m`, `elev_ground_min_m`, `elev_top_max_m` — the envelope |
| **any** | `roof_manual_any` — some part was manually post-processed (`roof_source` 6000); `roof_fallback_any` — some part is a LoD1 flat-box fallback, so part of the volume is a box |

### The two columns that exist because this is lossy

`function` comes from the largest part, which is **arbitrary when the parts
disagree** — and a hospital wing tagged differently from its main block is
exactly the case step 04 must not get wrong. So:

* **`n_functions`** — how many distinct AdV codes the parts carry
* **`functions_all`** — the full semicolon-separated list, filled only where
  `n_functions > 1`

A mixed-use building is therefore *visible as mixed* rather than silently
becoming whatever its biggest piece happens to be. Same idea for
`height_top_avg_m`: the area-weighted mean height alongside the max, since a
tall thin tower attached to a long low hall is badly described by either alone.

`is_multipart` flags a dissolved outline whose pieces do not touch — 0.42 % of
buildings, worth knowing before treating the geometry as one structure.

In [11]:
# --- representative part per building (largest ground area) ------------------
parts_b = bld.copy()
parts_b['_is_uuid'] = parts_b['gml_id'].str.startswith('UUID_')
parts_b['_wh'] = parts_b['height_top_m'] * parts_b['area_m2']
parts_b['_manual'] = parts_b['roof_source'] == 6000

# `created_on` and `plan_acquired_on` are date STRINGS, and plan_acquired_on has
# ~18.6k nulls. A groupby min() over an object column holding both str and NaN
# raises TypeError as soon as one group mixes them - 9,714 buildings do. Filling
# with a sentinel that sorts after every real date keeps the aggregation
# vectorised; a per-group lambda would work too but takes minutes over 869k
# groups. The sentinel is swapped back for null afterwards.
_LATE = '9999-12-31'
parts_b['_created'] = parts_b['created_on'].fillna(_LATE)
parts_b['_plan'] = parts_b['plan_acquired_on'].fillna(_LATE)

rep = (parts_b.loc[parts_b.groupby('alkis_id')['area_m2'].idxmax()]
       .set_index('alkis_id'))
print(f'  ..  representative part chosen for {len(rep):,} buildings')

agg = parts_b.groupby('alkis_id').agg(
    n_parts=('gml_id', 'size'),
    n_uuid=('_is_uuid', 'sum'),
    volume_3d_m3=('volume_3d_m3', 'sum'),
    volume_old_m3=('volume_old_m3', 'sum'),
    roof_area_m2=('roof_area_m2', 'sum'),
    n_surfaces=('n_surfaces', 'sum'),
    n_roof_faces=('n_roof_faces', 'sum'),
    height_top_max_m=('height_top_m', 'max'),
    height_eaves_max_m=('height_eaves_m', 'max'),
    elev_ground_min_m=('elev_ground_m', 'min'),
    elev_top_max_m=('elev_top_m', 'max'),
    n_roof_clipped=('n_roof_clipped', 'sum'),
    roof_manual_any=('_manual', 'max'),
    roof_fallback_any=('roof_is_fallback', 'max'),
    n_functions=('function', 'nunique'),
    created_on=('_created', 'min'),
    plan_acquired_on=('_plan', 'min'),
    name=('name', 'first'),          # GroupBy.first skips nulls
    _sum_part_area=('area_m2', 'sum'),
    _sum_wh=('_wh', 'sum'),
)
agg['height_top_avg_m'] = (agg['_sum_wh'] / agg['_sum_part_area']).round(3)
agg = agg.drop(columns=['_sum_wh', '_sum_part_area'])

# undo the sentinel: a building whose every part lacked a date keeps a null
for _c in ('created_on', 'plan_acquired_on'):
    _n = int((agg[_c] == _LATE).sum())
    agg.loc[agg[_c] == _LATE, _c] = None
    print(f'  ..  {_c}: {_n:,} buildings have no date on any part -> null')

# The full function list, only where the parts actually disagree - joining
# strings over all 869k groups would be wasted work.
mixed = agg.index[agg['n_functions'] > 1]
print(f'  ..  buildings whose parts disagree on `function`: {len(mixed):,} '
      f'({100 * len(mixed) / len(agg):.2f} %)')
agg['functions_all'] = (
    parts_b[parts_b['alkis_id'].isin(mixed)]
    .groupby('alkis_id')['function']
    .apply(lambda s: ';'.join(sorted(set(s))))
)

t0 = time.perf_counter()
print(f'  ..  dissolving {len(parts_b):,} part footprints by alkis_id ...', flush=True)
outl = parts_b[['alkis_id', 'geometry']].dissolve(by='alkis_id')
print(f'  ok  {len(outl):,} outlines  [{time.perf_counter() - t0:,.0f}s]')

byb = gpd.GeoDataFrame(agg.join(outl), geometry='geometry', crs=bld.crs)
# area from the DISSOLVED shape, not the sum of parts, so overlaps are not
# counted twice. The gap between the two is itself informative.
byb['area_m2'] = byb.geometry.area.round(2)
byb['is_multipart'] = (byb.geometry.geom_type == 'MultiPolygon')
byb['volume_ratio'] = (byb['volume_old_m3'] /
                       byb['volume_3d_m3'].replace(0, np.nan)).round(4)
for c in ('volume_3d_m3', 'volume_old_m3', 'roof_area_m2'):
    byb[c] = byb[c].round(1)
for c in ('function', 'roof_shape', 'roof_code', 'ags', 'city', 'street',
          'house_number'):
    byb[c] = rep[c]
byb = byb.reset_index()

miss = [c for c in ALKIS_BUILDING_COLS if c not in byb.columns]
if miss:
    raise AssertionError(f'ALKIS_BUILDING_COLS names missing columns: {miss}')
byb = byb[ALKIS_BUILDING_COLS]
require_unique(byb, 'alkis_id', 'alkis by building')
# `functions_all` is legitimately all-NULL here - see ALKIS_ALLOW_EMPTY_COLS.
assert_no_empty_columns(byb.drop(columns=ALKIS_ALLOW_EMPTY_COLS),
                        'alkis by building')

print()
print(f'  ..  parts {len(bld):,} -> buildings {len(byb):,} '
      f'({len(bld) / len(byb):.2f} parts each)')
print(f'  ..  volume conserved? parts {bld["volume_3d_m3"].sum() / 1e6:,.3f} vs '
      f'buildings {byb["volume_3d_m3"].sum() / 1e6:,.3f} million m3')
_dv = abs(bld['volume_3d_m3'].sum() - byb['volume_3d_m3'].sum())
if _dv > 1.0:
    raise AssertionError(f'volume not conserved by the aggregation: {_dv:,.1f} m3 lost')
print(f'  ok  volume conserved to {_dv:.3f} m3')
print(f'  ..  footprint area: sum of parts {parts_b["area_m2"].sum() / 1e6:,.2f} vs '
      f'dissolved {byb["area_m2"].sum() / 1e6:,.2f} km2-ish '
      f'({100 * (1 - byb["area_m2"].sum() / parts_b["area_m2"].sum()):+.2f} % '
      f'removed as part overlap)')
print(f'  ..  multipart outlines : {int(byb["is_multipart"].sum()):,} '
      f'({100 * byb["is_multipart"].mean():.2f} %)')
print(f'  ..  mixed-function     : {int((byb["n_functions"] > 1).sum()):,}')
print(f'  ..  named buildings    : {int(byb["name"].notna().sum()):,} '
      f'({100 * byb["name"].notna().mean():.2f} %)')

  ..  representative part chosen for 869,316 buildings


  ..  created_on: 0 buildings have no date on any part -> null


  ..  plan_acquired_on: 2 buildings have no date on any part -> null
  ..  buildings whose parts disagree on `function`: 0 (0.00 %)
  ..  dissolving 1,332,029 part footprints by alkis_id ...


  ok  869,316 outlines  [49s]


  ok  alkis by building.alkis_id: unique and non-null (869,316)


  ok  alkis by building: 31 columns, none empty

  ..  parts 1,332,029 -> buildings 869,316 (1.53 parts each)
  ..  volume conserved? parts 677.270 vs buildings 677.270 million m3
  ok  volume conserved to 0.000 m3
  ..  footprint area: sum of parts 90.03 vs dissolved 90.02 km2-ish (+0.01 % removed as part overlap)
  ..  multipart outlines : 3,654 (0.42 %)
  ..  mixed-function     : 0
  ..  named buildings    : 5,029 (0.58 %)


In [12]:
# --- write the building-level layer ------------------------------------------
if ALKIS_BY_BUILDING_FILE.exists():
    try:
        ALKIS_BY_BUILDING_FILE.unlink()
    except PermissionError as e:
        raise RuntimeError(
            f'{ALKIS_BY_BUILDING_FILE.name} is locked - close it in QGIS or in '
            f'another notebook kernel. Original error: {e}'
        ) from None

print(f'Writing {len(byb):,} rows x {len(byb.columns)} columns to '
      f'{ALKIS_BY_BUILDING_FILE.name} ...', flush=True)
t0 = time.perf_counter()
byb.to_file(ALKIS_BY_BUILDING_FILE, layer='buildings', driver='GPKG')
print(f'  ok  {ALKIS_BY_BUILDING_FILE.stat().st_size / 1e6:,.1f} MB  '
      f'[{time.perf_counter() - t0:,.0f}s]')

chk = gpd.read_file(ALKIS_BY_BUILDING_FILE, layer='buildings', rows=6)
print(f'  ok  read back: CRS {chk.crs}')
print()
print(chk[['alkis_id', 'n_parts', 'area_m2', 'volume_3d_m3', 'volume_old_m3',
           'volume_ratio', 'function', 'n_functions', 'roof_shape',
           'height_top_max_m']].to_string(index=False))

# --- and the throwaway shapefile, from the same dissolve --------------------
# Shapefile field names cap at 10 characters, so this carries the short set.
EXPERIMENTAL_DIR.mkdir(parents=True, exist_ok=True)
shp = byb[['alkis_id', 'n_parts', 'n_uuid', 'area_m2', 'geometry']].copy()
shp['vol_3d'] = byb['volume_3d_m3']
shp['vol_old'] = byb['volume_old_m3']
shp['vol_ratio'] = byb['volume_ratio']
shp['func'] = byb['function']
for ext in ('.shp', '.shx', '.dbf', '.prj', '.cpg'):
    ALKIS_OUTLINES_SHP.with_suffix(ext).unlink(missing_ok=True)
shp.to_file(ALKIS_OUTLINES_SHP, driver='ESRI Shapefile')
_tot = sum(p.stat().st_size for p in
           ALKIS_OUTLINES_SHP.parent.glob(ALKIS_OUTLINES_SHP.stem + '.*'))
print(f'\n  ok  {ALKIS_OUTLINES_SHP.name} + sidecars, {_tot / 1e6:,.1f} MB '
      f'(QGIS convenience copy; the GeoPackage above is authoritative)')

Writing 869,316 rows x 32 columns to 03_alkis_by_building.gpkg ...


  ok  424.7 MB  [15s]
  ok  read back: CRS EPSG:25832

        alkis_id  n_parts  area_m2  volume_3d_m3  volume_old_m3  volume_ratio   function  n_functions   roof_shape  height_top_max_m
DENIAL01000000Fg        1     4.00          14.0           14.0        1.0000 51002_1250            1 PolyFlatRoof             3.500
DENIAL01000000Fh        1     4.00          14.0           14.0        1.0000 51002_1250            1 PolyFlatRoof             3.500
DENIAL01000002A0        2   115.68         673.3          840.7        1.2486 31001_1000            1    GableRoof             7.404
DENIAL01000002A1        1   212.80         931.4          931.4        1.0000 31001_2000            1 PolyFlatRoof             4.377
DENIAL01000002A3        1   199.08         882.2         1026.3        1.1633 31001_2000            1    GableRoof             5.155
DENIAL01000002A4        1    44.06         121.0          121.0        1.0000 31001_2000            1 PolyFlatRoof             2.746



  ok  alkis_outlines.shp + sidecars, 432.1 MB (QGIS convenience copy; the GeoPackage above is authoritative)


## 9. Where this leaves us

**`03_alkis_buildings.gpkg`** — 1,332,029 flat footprints, one per LoD2 part
(1,385,265 before the sliver filter).
Authoritative, lossless, `alkis_id` on every row.

**`03_alkis_by_building.gpkg`** — the same data as 869,316 real ALKIS buildings,
with `n_functions` / `functions_all` marking where the collapse had to pick a
winner. Convenient for the POI join and for anything that thinks in buildings.

Notebook 04 can take either; the part layer is the one to fall back on when a
building-level answer looks wrong.

What was established getting here:

* `gml_id` is a **part** key (1,385,279), not a building key. `externRef`'s `$$$`
  tail is the ALKIS object id, present on 100 % of rows for both the `DENILD…`
  (81 %) and `UUID…` (19 %) forms, at 1.59 parts per object, max 228.
* Surfaces are **classified by Z**, verified against CityGML's own labels on
  10,803 parts: every `GroundSurface` found, zero false positives, footprint
  areas identical to `+0.000000 %`. 14 parts (11 ALKIS objects, 9 of them
  masts and towers) have no usable ground surface and are absent from both
  outputs; section 5 lists them.
* **The prism sum is exact per face** — checked on 4,380 parts against an
  independent plane-fit integral (median deviation 4e-7) and a 0.25 m grid
  integral (within ±2 % for 95 % of the 3,780 parts above 5 m², 99 % above
  20 m²) — **once each face is clipped to its
  part's footprint**. LGLN's multi-part hip roofs spill over neighbouring
  parts; unclipped, 16,704 buildings were overstated by a median 13 %, 4.47 M m³
  or 0.66 % of the region in total. `n_roof_clipped` marks where the clip acted.
* **`height_eaves_m` is a mean** (LGLN: *Mittlere Traufhöhe*), `roof_source`
  6000 is the manually checked subset, `roof_is_fallback` marks LoD1 flat
  boxes, and `roof_code`/`roof_shape` are two different classifications. The
  LGLN Produkt- und Formatbeschreibung LoD2 (2018), the AdV Datenformatbeschreibung
  LoD2-DE (2025) and LGLN's 3D-Kundentag slides (Wichmann, 2019: recognition
  rate, post-processing shares, *Mittlere Traufhöhe*) are the sources; config.py
  quotes them.
* **`volume_old_m3` overstates volume** and the bias is *directional* — exact
  for flat roofs, tens of percent for pitched. Flat roofs skew commercial and industrial
  while gables skew residential, so the original method inflates residential
  stock by roughly a quarter relative to commercial. In a volume-proportional
  redistribution that misplaces workers and retail demand into housing; it
  changes *where* demand lands, not merely the scale.

Still open, deliberately:

* **no volume threshold.** The distribution is in the output; pick the number
  from it rather than inheriting `>= 1 m³`.
* **no merging** of touching or overlapping polygons.
* **11.57 % of rows are not `AX_Gebaeude`.** The `function` prefix splits
  `31001` 768,756 (88.4 %), `51009` 94,582 (10.9 %), then `51002`/`51003`/
  `51001`/`51006`/`51007`. The `51xxx` classes are other structures — this is
  where canopies live. If the reference tables only cover `31001_*`, roughly
  100,000 rows fall out of the step 04 join, so check that before trusting a
  coverage figure.
* **no function labels.** `function` is the raw AdV code. Joining
  `building_function_codelist_de_en.csv` (301 codes) and
  `alkis_building_activity_map.xlsx` (280) belongs in notebook 04 — and the
  question that matters there is whether all 88 codes occurring in this region
  appear in the activity map, since a code missing there yields no activities and
  those buildings would drop out of the redistribution silently.